# [재학습 · 2026-10-05 회계 수정] 8장 한계#3 DDQN 라우터 실험

원본 §13-I(DQN 매칭 라우터)와 §22(DDQN 매칭 라우터)를 **회계 수정 후 환경**에서 다시 학습해 비교한다.
- 하위 풀(v4 β 전문가 4개)은 기존 모델을 그대로 쓴다(수정 전 보상으로 학습, 논문 한계#11과 같은 조건).
- 라우터만 다시 학습: DQN 매칭 + DDQN 매칭, 각 20k 스텝, 같은 시드.
- 비교: valid/test에서 β 선택 분포, α=0.01 스무딩 성과, β=30 단독과의 일치 여부.

모델·CSV에 `_fix1004`를 붙였다. 위에서부터 **순서대로 전부 실행**(CPU).


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q stable-baselines3[extra] gymnasium h5py pyarrow


In [ ]:
import warnings
import glob
import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from dataclasses import dataclass

import gymnasium as gym
from gymnasium import spaces
from stable_baselines3 import PPO, DQN
from stable_baselines3.common.env_checker import check_env
from stable_baselines3.common.callbacks import EvalCallback
from stable_baselines3.common.monitor import Monitor

warnings.filterwarnings('ignore')
np.random.seed(42)
plt.rcParams['axes.unicode_minus'] = False

DRIVE_ROOT = Path('/content/drive/MyDrive/졸업프로젝트')
DATA_DIR   = DRIVE_ROOT / 'data'
CRYPTO_DIR = DATA_DIR / 'crypto'
FEAT_DIR   = CRYPTO_DIR / 'features_5m'
MODEL_DIR  = DRIVE_ROOT / 'models'
POOL_DIR   = MODEL_DIR / 'crypto_hrl_pool'

RL_EMB_H5      = CRYPTO_DIR / 'crypto_rl_embeddings.h5'
ORIG_PPO_PATH  = MODEL_DIR / 'crypto_ppo_portfolio.zip'   # crypto_rl_trading_colab.ipynb 산출물(있으면 비교에 포함)

EMB_DIM    = 64
TC         = 0.001    # 거래비용 (Binance taker fee 기준)
BUCKET_SEC = 1800     # 30분 버킷

RL_TRAIN_START = '2025-01-01'
RL_TRAIN_END   = '2025-12-31'
RL_TEST_START  = '2026-01-01'
RL_TEST_END    = '2099-12-31'   # 열린 구간

# --- Stage A/B/C 하이퍼파라미터 ---
N_LABELS              = 5          # 레짐 라벨 수 (bear/pullback/sideways/rally/bull)
RISK_THRESHOLD        = 0.2        # EarnHFT θ — 상하위 라벨 경계 분위수 폭
LOWPASS_WINDOW        = 48         # 레짐 분할 저역통과 윈도우 = 1일(30분×48)
POOL_BETAS            = [-90, -30, 30, 90]   # Stage B 선호도 스윕 (논문 [-90,-10,30,100] 근사)
POOL_TIMESTEPS_EACH   = 100_000    # 풀 에이전트 1개당 학습 스텝 (원본 300k와 총량 비슷하게)
ROUTE_BUCKETS         = 48         # Stage C 라우팅 단위 = 1일
MACRO_WINDOW           = 96        # 라우터 상태에 포함할 최근 버킷 수 (=2일)
ROUTER_TOTAL_TIMESTEPS = 20_000    # 라우터(=일 단위) 학습 스텝

assert RL_EMB_H5.exists(), 'crypto_rl_embeddings.h5 없음 — crypto_hybrid_model_colab.ipynb 먼저 실행'
print('경로 확인 완료')


In [ ]:
print('crypto_rl_embeddings.h5 로드 중...')
with h5py.File(RL_EMB_H5, 'r') as f:
    symbols_raw = f['symbols'][:]
    ts_raw      = f['timestamps'][:]     # epoch seconds (UTC)
    embs_all    = f['embeddings'][:]     # (N, 64)

symbols_list = [s.decode() for s in symbols_raw]
ts_list      = ts_raw.astype(np.int64).tolist()

# (symbol, 30분 버킷) → 임베딩 인덱스
emb_index  = {}
sym_ts_set = {}   # 심볼별로 임베딩 추출에 실제 쓰인 초 단위 timestamp 집합 (수익률 정확 매칭용)
for i, (s, t) in enumerate(zip(symbols_list, ts_list)):
    emb_index[(s, t // BUCKET_SEC)] = i
    sym_ts_set.setdefault(s, set()).add(t)

symbols  = sorted(set(symbols_list))
N_ASSETS = len(symbols)
print(f'임베딩: {embs_all.shape} | 심볼: {N_ASSETS}개 | 인덱스: {len(emb_index):,}개')


In [ ]:
print('features_5m parquet에서 RetTarget_6b 로드 중...')
files = sorted(glob.glob(str(FEAT_DIR / '*.parquet')))
assert files, f'피처 parquet 없음: {FEAT_DIR}'

rl_start_ts = pd.Timestamp(RL_TRAIN_START, tz='UTC')
ret_index   = {}

for fp in files:
    sym = Path(fp).stem
    valid_ts = sym_ts_set.get(sym)
    if not valid_ts:
        continue
    df = pd.read_parquet(fp, columns=['datetime', 'RetTarget_6b'])
    df = df[df['datetime'] >= rl_start_ts]
    epoch = df['datetime'].values.astype('int64') // 10**9   # ts_list와 동일 단위(초)
    y     = df['RetTarget_6b'].values
    # 임베딩 추출에 실제 쓰인 timestamp만 매칭 — 같은 버킷 내 다른 5분봉과 섞이지 않도록
    for t, r in zip(epoch, y):
        t = int(t)
        if t in valid_ts and not np.isnan(r):
            ret_index[(sym, t // BUCKET_SEC)] = float(r)

all_buckets = sorted(set(t // BUCKET_SEC for t in ts_list))
print(f'수익률 인덱스: {len(ret_index):,}개 | 전체 30분 버킷: {len(all_buckets):,}개')


In [ ]:
# [2026-10-04 회계 수정] 가격 변동 후 실제 보유 비중(drift)·거래 가능 종목 마스크.
# 이전 버전은 수익 반영 뒤에도 직전 '목표' 비중을 보유 비중으로 저장해, 가격 변동으로 틀어진 비중을
# 되돌리는 재조정 거래(등가중 포함)의 비용이 0으로 계산됐다.
def _drift_weights(w, rets):
    """수익 반영 뒤 실제 보유 비중 w_i(1+r_i)/(1+wᵀr). 합<1이면 나머지는 현금(수익률 0)으로 남는다."""
    w = np.asarray(w, dtype=np.float64)
    g = 1.0 + float(np.dot(w, rets))
    if g <= 1e-12:
        return w.astype(np.float32)
    return (w * (1.0 + np.asarray(rets, dtype=np.float64)) / g).astype(np.float32)


def _mask_untradable(w, tradable, renorm=True):
    """수익률 데이터가 없는(상장 전·상폐 후) 종목 비중을 0으로. renorm=False면 빠진 몫은 현금으로 남는다."""
    w = np.where(tradable, np.asarray(w, dtype=np.float64), 0.0)
    s = w.sum()
    return (w / s if renorm and s > 1e-12 else w).astype(np.float32)


class CryptoPortfolioEnv(gym.Env):
    """
    크립토 44종목 개별 자산 기반 포트폴리오 운용 환경 (클러스터링 없음)

    State:  종목별 임베딩(N×64) + 현재 비중(N) → N*64+N dim
    Action: 종목별 목표 포트폴리오 비중 (N-dim, Softmax 정규화)
    Reward: 30분 포트폴리오 수익률(RetTarget_6b, 이미 forward-looking) − 거래비용
    스텝 = 30분 버킷(BUCKET_SEC) 단위 — emb_index/ret_index 모두 (symbol, bucket) 키.
    """
    metadata = {'render_modes': []}

    def __init__(self, buckets, symbols, emb_index, embs_all, ret_index,
                 emb_dim=64, tc=0.001):
        super().__init__()
        self.buckets    = buckets
        self.symbols    = symbols
        self.n_assets   = len(symbols)
        self.emb_index  = emb_index
        self.embs_all   = embs_all
        self.ret_index  = ret_index
        self.emb_dim    = emb_dim
        self.tc         = tc

        obs_dim = self.n_assets * emb_dim + self.n_assets
        self.observation_space = spaces.Box(-np.inf, np.inf, shape=(obs_dim,), dtype=np.float32)
        self.action_space      = spaces.Box(-3.0, 3.0, shape=(self.n_assets,), dtype=np.float32)

    def _asset_emb(self, bucket):
        """종목별 임베딩 — 해당 버킷에 데이터 없으면 0-vector(상장 전/상폐 등)"""
        embs = []
        for sym in self.symbols:
            idx = self.emb_index.get((sym, bucket))
            embs.append(self.embs_all[idx] if idx is not None else np.zeros(self.emb_dim, dtype=np.float32))
        return np.concatenate(embs)  # (N*64,)

    def _asset_ret(self, bucket):
        """종목별 30분 수익률 — 없으면 0(그 자산은 보유해도 손익 없음으로 취급)"""
        return np.array([self.ret_index.get((sym, bucket), 0.0) for sym in self.symbols], dtype=np.float32)

    def _tradable(self, bucket):
        """해당 버킷에 수익률 데이터가 있는(거래 가능한) 종목 마스크"""
        return np.array([(sym, bucket) in self.ret_index for sym in self.symbols])

    def _get_obs(self):
        b = self.buckets[self.t]
        return np.concatenate([self._asset_emb(b), self.weights]).astype(np.float32)

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.t               = 0
        self.weights         = _mask_untradable(np.ones(self.n_assets, dtype=np.float32),
                                          self._tradable(self.buckets[0]))  # 거래 가능 종목 등가중에서 시작
        self.portfolio_value = 1.0
        self.value_history   = [1.0]
        self.turnover_history = []
        return self._get_obs(), {}

    def step(self, action):
        # Softmax로 비중 정규화
        action  = np.exp(action - action.max())
        action  = action / action.sum()
        action  = _mask_untradable(action, self._tradable(self.buckets[self.t]))  # 거래 불가 종목 제외

        # 거래비용
        turnover = float(np.abs(action - self.weights).sum())  # 비중 변화량(진단용)
        tc_cost  = self.tc * turnover

        # RetTarget_6b는 현재 버킷 자체에 forward 수익률이 이미 인코딩되어 있음
        b          = self.buckets[self.t]
        asset_rets = self._asset_ret(b)
        port_ret   = float((action * asset_rets).sum())

        self.portfolio_value *= (1 + port_ret - tc_cost)
        self.weights           = _drift_weights(action, asset_rets)  # 가격 변동 후 실제 보유 비중
        self.value_history.append(self.portfolio_value)
        self.turnover_history.append(turnover)
        self.t                += 1

        done = (self.t >= len(self.buckets) - 1)
        reward = float(port_ret - tc_cost)

        info = {'portfolio_value': self.portfolio_value, 'port_ret': port_ret, 'turnover': turnover, 'tc_cost': tc_cost}
        return self._get_obs(), reward, done, False, info


In [ ]:
def bucket_to_ts(b):
    return pd.Timestamp(b * BUCKET_SEC, unit='s', tz='UTC')

def bucket_to_date(b):
    return bucket_to_ts(b).strftime('%Y-%m-%d')

train_buckets = [b for b in all_buckets if RL_TRAIN_START <= bucket_to_date(b) <= RL_TRAIN_END]
test_buckets  = [b for b in all_buckets if RL_TEST_START  <= bucket_to_date(b) <= RL_TEST_END]
print(f'학습 기간: {bucket_to_ts(train_buckets[0])} ~ {bucket_to_ts(train_buckets[-1])} ({len(train_buckets):,}스텝)')
print(f'테스트 기간: {bucket_to_ts(test_buckets[0])} ~ {bucket_to_ts(test_buckets[-1])} ({len(test_buckets):,}스텝)')

env_kwargs = dict(
    symbols=symbols, emb_index=emb_index, embs_all=embs_all,
    ret_index=ret_index, emb_dim=EMB_DIM, tc=TC,
)

test_env = CryptoPortfolioEnv(test_buckets, **env_kwargs)

print('\n환경 유효성 검사...')
check_env(CryptoPortfolioEnv(train_buckets[:100], **env_kwargs), warn=True)
print('OK')


In [ ]:
TOP_K = 10   # 종목 유니버스(44) 중 매 스텝 보유 가능한 최대 종목 수 (FreQuant top-|G| 이식)

class SparseGatedCryptoPortfolioEnv(CryptoPortfolioEnv):
    """
    FreQuant(top-K 희소 선택) + DeepClair(전용 리스크게이트 ρ) 결합 저수준 환경.

    Action: (N+1,)-dim — 앞 N개는 종목별 confidence logit(FreQuant Eq.13-14의 h_i 역할),
            마지막 1개는 리스크자산 총 노출 게이트 로짓(DeepClair의 ρ_t 역할).
    변환:   top-K confidence logit만 골라 그 안에서 softmax(FreQuant Eq.14) →
            sigmoid(게이트)를 곱해 최종 종목별 비중. 나머지는 암묵적 현금(수익률 0).
    관측·NAV 회계는 CryptoPortfolioEnv와 동일. step_with_weights()는 목표 비중을 직접
    받아 NAV를 갱신하는 보조 메서드 — §13-G 관성 스무딩에서 (top-K 변환을 다시 태우지 않고)
    이미 블렌딩된 비중을 그대로 반영하기 위해 필요하다.
    """

    def __init__(self, *args, top_k=TOP_K, **kwargs):
        self.top_k = top_k
        super().__init__(*args, **kwargs)
        self.action_space = spaces.Box(-3.0, 3.0, shape=(self.n_assets + 1,), dtype=np.float32)

    def action_to_weights(self, action):
        action = np.asarray(action, dtype=np.float64)
        asset_logits, gate_logit = action[:-1], action[-1]
        asset_logits = np.where(self._tradable(self.buckets[self.t]), asset_logits, -np.inf)  # 거래 불가 종목 제외
        k = min(self.top_k, self.n_assets)
        top_idx = np.argpartition(asset_logits, -k)[-k:]
        sel = asset_logits[top_idx]
        sel = np.exp(sel - sel.max())
        sel = sel / sel.sum()
        risky_w = np.zeros(self.n_assets, dtype=np.float32)
        risky_w[top_idx] = sel
        rho = float(1.0 / (1.0 + np.exp(-gate_logit)))
        return (rho * risky_w).astype(np.float32), rho, top_idx

    def reset(self, seed=None, options=None):
        obs, info = super().reset(seed=seed, options=options)
        self.rho_history = []
        self.selection_history = []
        return obs, info

    def step_with_weights(self, target_w, extra_info=None):
        target_w = _mask_untradable(target_w, self._tradable(self.buckets[self.t]), renorm=False)  # 빠진 몫은 현금
        turnover = float(np.abs(target_w - self.weights).sum())
        tc_cost  = self.tc * turnover
        b          = self.buckets[self.t]
        asset_rets = self._asset_ret(b)
        port_ret   = float((target_w * asset_rets).sum())

        self.portfolio_value *= (1 + port_ret - tc_cost)
        self.weights           = _drift_weights(target_w, asset_rets)  # 가격 변동 후 실제 보유 비중
        self.value_history.append(self.portfolio_value)
        self.turnover_history.append(turnover)
        self.t                += 1

        done = (self.t >= len(self.buckets) - 1)
        reward = float(port_ret - tc_cost)
        info = {'portfolio_value': self.portfolio_value, 'port_ret': port_ret,
                'turnover': turnover, 'tc_cost': tc_cost}
        if extra_info:
            info.update(extra_info)
            if 'rho' in extra_info:
                self.rho_history.append(extra_info['rho'])
            if 'selected' in extra_info:
                self.selection_history.append(extra_info['selected'])
        return self._get_obs(), reward, done, False, info

    def step(self, action):
        target_w, rho, top_idx = self.action_to_weights(action)
        return self.step_with_weights(target_w, extra_info={'rho': rho, 'selected': top_idx.tolist()})


# 스모크 테스트
_sg_env = SparseGatedCryptoPortfolioEnv(test_buckets[:200], top_k=TOP_K, **env_kwargs)
check_env(_sg_env, warn=True)
obs, _ = _sg_env.reset()
a = np.random.randn(_sg_env.n_assets + 1).astype(np.float32)
obs, r, done, _, info = _sg_env.step(a)
print(f"action space: {_sg_env.action_space.shape} | 선택된 종목 수: {len(info['selected'])} | "
      f"rho: {info['rho']:.3f} | reward: {r:.5f}")

In [ ]:
class MacroRouterEnvV4(gym.Env):
    """§6 MacroRouterEnv와 동일 로직, 저수준 환경만 SparseGatedCryptoPortfolioEnv로 교체.

    episode_progress_mode: 관측값 마지막 성분(t/len(buckets), '에피소드 내 진행률').
      - 'real'(기본값, §13-C/§13-D 학습에 사용): 실제 진행률을 그대로 넘김 — 원 EarnHFT
        라우터 관측 설계를 그대로 재현하지만, 실거래에는 '에피소드 끝'이 없어
        policy_v4.py는 이 값을 상수 0.5로 근사한다(라이브가 학습 때 없던 입력 분포를
        받는 구조적 불일치, 모듈 docstring 한계 3).
      - 'constant': 학습 중에도 항상 0.5로 고정 — 라이브가 실제로 보게 될 입력 분포와
        학습 분포를 일치시킨 '매칭' 라우터(§13-I)용.
    """
    metadata = {'render_modes': []}

    def __init__(self, buckets, pool_models, route_buckets=48, macro_window=96, top_k=TOP_K,
                 episode_progress_mode='real', **env_kwargs):
        super().__init__()
        self.pool_models           = list(pool_models)
        self.route_buckets         = route_buckets
        self.macro_window          = macro_window
        self.episode_progress_mode = episode_progress_mode
        self.inner           = SparseGatedCryptoPortfolioEnv(buckets=buckets, top_k=top_k, **env_kwargs)
        self.action_space      = spaces.Discrete(len(self.pool_models))
        self.observation_space = spaces.Box(-np.inf, np.inf, shape=(macro_window + 4,), dtype=np.float32)
        self._nav_history = []

    def _macro_obs(self):
        hist = self._nav_history[-self.macro_window:]
        if len(hist) < self.macro_window:
            hist = [1.0] * (self.macro_window - len(hist)) + hist
        recent = np.array(hist, dtype=np.float32)
        ret = np.diff(recent) if len(recent) > 1 else np.zeros(1, dtype=np.float32)
        if self.episode_progress_mode == 'constant':
            episode_progress = 0.5
        else:
            episode_progress = self.inner.t / max(len(self.inner.buckets) - 1, 1)
        stats = np.array([
            recent[-1] - recent[0],
            float(ret.std()),
            float(ret.mean()),
            episode_progress,
        ], dtype=np.float32)
        return np.concatenate([recent, stats]).astype(np.float32)

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.inner.reset(seed=seed)
        self._nav_history = []
        return self._macro_obs(), {}

    def step(self, action):
        model = self.pool_models[int(action)]
        block_reward = 0.0
        done_inner = False
        for _ in range(self.route_buckets):
            obs = self.inner._get_obs()
            a, _ = model.predict(obs, deterministic=True)
            _, r, done_inner, _, info = self.inner.step(a)
            block_reward += r
            self._nav_history.append(info['portfolio_value'])
            if done_inner:
                break
        done = done_inner or self.inner.t >= len(self.inner.buckets) - 1
        info = {'portfolio_value': self.inner.portfolio_value}
        return self._macro_obs(), block_reward, done, False, info


# 스모크 테스트 — pool_models_v4가 아직 없는 세션(§13-C를 건너뛰고 §13-로더만 나중에 실행하는
# 복구 경로)에서도 이 셀이 클래스 정의만은 항상 성공하도록 방어.
if 'pool_models_v4' in globals() and pool_models_v4:
    _router_probe_v4 = MacroRouterEnvV4(
        buckets=train_buckets[:500], pool_models=list(pool_models_v4.values()),
        route_buckets=ROUTE_BUCKETS, macro_window=MACRO_WINDOW, top_k=TOP_K, **env_kwargs,
    )
    obs, _ = _router_probe_v4.reset()
    print('[v4] router obs shape:', obs.shape)
    obs, r, done, _, info = _router_probe_v4.step(0)
    print('[v4] 1스텝 후 reward:', r, 'portfolio_value:', info['portfolio_value'])
else:
    print('MacroRouterEnvV4 클래스 정의 완료 (pool_models_v4 없음 — 스모크 테스트 생략, §13-로더 실행 후 계속 진행)')


In [ ]:
# §13-K-A — valid 구간 정의 (v3 §12와 동일 분할 재사용: train 1~9월 / valid 10~12월)
VALID_START, VALID_END = '2025-10-01', '2025-12-31'
valid_buckets = [b for b in train_buckets if VALID_START <= bucket_to_date(b) <= VALID_END]
print(f'valid 구간: {bucket_to_ts(valid_buckets[0])} ~ {bucket_to_ts(valid_buckets[-1])} ({len(valid_buckets):,}스텝)')

def perf_num(values, steps_per_year=48 * 365):
    """수치형 성과 — 논문 표에 바로 옮길 수 있게 float로 반환"""
    values = np.asarray(values, dtype=np.float64)
    rets = np.diff(values) / values[:-1]
    return {'ret': values[-1] / values[0] - 1,
            'sharpe': rets.mean() / (rets.std() + 1e-8) * np.sqrt(steps_per_year),
            'mdd': ((values / np.maximum.accumulate(values)) - 1).min()}

compute_perf = perf_num   # 원본 셀들이 부르는 이름 유지

POOL_DIR_V4 = MODEL_DIR / 'crypto_hrl_pool_v4'
pool_models_v4 = {b: PPO.load(str(POOL_DIR_V4 / f'agent_beta_{b}.zip')) for b in POOL_BETAS}
print('v4 풀 로드:', list(pool_models_v4.keys()))


In [ ]:
%%time
# §13-I-A — 학습 때도 episode_progress를 0.5로 고정한 "매칭" 라우터 (풀 에이전트는 v4를 그대로 재사용)
router_env_v4_matched = Monitor(MacroRouterEnvV4(
    buckets=train_buckets, pool_models=[pool_models_v4[b] for b in POOL_BETAS],
    route_buckets=ROUTE_BUCKETS, macro_window=MACRO_WINDOW, top_k=TOP_K,
    episode_progress_mode='constant', **env_kwargs,
))

router_v4_matched = DQN(
    'MlpPolicy', router_env_v4_matched,
    learning_rate=1e-3, buffer_size=20_000, learning_starts=500,
    batch_size=64, gamma=0.95, train_freq=1, target_update_interval=200,
    exploration_fraction=0.3, exploration_final_eps=0.05,
    policy_kwargs=dict(net_arch=[128, 64]),
    verbose=1, device='cpu', seed=42,
)
router_v4_matched.learn(total_timesteps=ROUTER_TOTAL_TIMESTEPS, progress_bar=True)

router_v4_matched_path = MODEL_DIR / 'crypto_hrl_router_v4_matched_fix1004.zip'
router_v4_matched.save(str(router_v4_matched_path))
print(f'[v4] 매칭 라우터 저장: {router_v4_matched_path.name}')


In [ ]:
import torch as th
from torch.nn import functional as F


class DoubleDQN(DQN):
    """표준 DQN(SB3 기본, v2.7.1 기준 train() 로직)을 Double DQN 방식으로 오버라이드.
    다음 상태의 행동 선택은 online network(q_net)로, 그 행동의 가치 평가는
    target network(q_net_target)로 분리해 과대추정 편향을 교정한다(van Hasselt et al., 2016).
    나머지 로직(리플레이 샘플링, 손실, 최적화)은 SB3 DQN.train()과 동일하게 유지 — 알고리즘
    선택 하나만 바뀐 순수 ablation이 되도록 다른 건 일절 건드리지 않았다.
    """

    def train(self, gradient_steps: int, batch_size: int = 100) -> None:
        self.policy.set_training_mode(True)
        self._update_learning_rate(self.policy.optimizer)

        losses = []
        for _ in range(gradient_steps):
            replay_data = self.replay_buffer.sample(batch_size, env=self._vec_normalize_env)
            discounts = replay_data.discounts if replay_data.discounts is not None else self.gamma

            with th.no_grad():
                # Double DQN: 다음 행동은 online network(q_net)로 선택
                next_q_values_online = self.q_net(replay_data.next_observations)
                next_actions = next_q_values_online.argmax(dim=1, keepdim=True)
                # 그 행동의 가치는 target network(q_net_target)로 평가
                next_q_values_target = self.q_net_target(replay_data.next_observations)
                next_q_values = th.gather(next_q_values_target, dim=1, index=next_actions)
                target_q_values = replay_data.rewards + (1 - replay_data.dones) * discounts * next_q_values

            current_q_values = self.q_net(replay_data.observations)
            current_q_values = th.gather(current_q_values, dim=1, index=replay_data.actions.long())

            loss = F.smooth_l1_loss(current_q_values, target_q_values)
            losses.append(loss.item())

            self.policy.optimizer.zero_grad()
            loss.backward()
            th.nn.utils.clip_grad_norm_(self.policy.parameters(), self.max_grad_norm)
            self.policy.optimizer.step()

        self._n_updates += gradient_steps
        self.logger.record('train/n_updates', self._n_updates, exclude='tensorboard')
        self.logger.record('train/loss', np.mean(losses))


# 스모크 검증 — online/target망에 인위적으로 차이를 만들어, DoubleDQN의 타겟 계산이
# vanilla DQN(타겟망 단독)과 실제로 다른 값을 내는지 확인(로직이 진짜 Double 방식인지 검증).
# 실제 라우터 환경에 의존하지 않는 최소 합성 환경으로 로직만 독립적으로 검증한다.
class _TinyDiscreteEnv(gym.Env):
    metadata = {'render_modes': []}

    def __init__(self):
        super().__init__()
        self.observation_space = spaces.Box(-1.0, 1.0, shape=(6,), dtype=np.float32)
        self.action_space = spaces.Discrete(4)
        self.t = 0

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.t = 0
        return self.observation_space.sample(), {}

    def step(self, action):
        self.t += 1
        return self.observation_space.sample(), float(action == 2), self.t >= 20, False, {}


_smoke_router = DoubleDQN(
    'MlpPolicy', Monitor(_TinyDiscreteEnv()),
    learning_rate=1e-3, buffer_size=2000, learning_starts=100, batch_size=32,
    policy_kwargs=dict(net_arch=[32, 16]), verbose=0, device='cpu', seed=0,
)
with th.no_grad():
    for p in _smoke_router.q_net_target.parameters():
        p.add_(th.randn_like(p) * 0.5)
_dummy_obs = th.randn(5, _smoke_router.observation_space.shape[0])
with th.no_grad():
    _online_q = _smoke_router.q_net(_dummy_obs)
    _target_q = _smoke_router.q_net_target(_dummy_obs)
    _double_vals = th.gather(_target_q, dim=1, index=_online_q.argmax(dim=1, keepdim=True))
    _vanilla_vals, _ = _target_q.max(dim=1)
_diff = (_double_vals.flatten() - _vanilla_vals).abs().sum().item()
print(f'Double DQN 타겟값과 vanilla DQN 타겟값의 차이 합계: {_diff:.4f}')
assert _diff > 1e-6, 'online/target망이 우연히 같아서 차이가 안 남 — 재확인 필요'
print('[OK] DoubleDQN이 vanilla DQN과 다른 타겟값을 계산함 — 구현이 실제로 Double 방식으로 작동')

In [ ]:
%%time
assert 'pool_models_v4' in globals(), '§13-C(v4 풀 학습)를 먼저 실행하세요 — pool_models_v4 필요'

router_env_v4_ddqn = Monitor(MacroRouterEnvV4(
    buckets=train_buckets, pool_models=[pool_models_v4[b] for b in POOL_BETAS],
    route_buckets=ROUTE_BUCKETS, macro_window=MACRO_WINDOW, top_k=TOP_K,
    episode_progress_mode='constant', **env_kwargs,
))

router_v4_ddqn = DoubleDQN(
    'MlpPolicy', router_env_v4_ddqn,
    learning_rate=1e-3, buffer_size=20_000, learning_starts=500,
    batch_size=64, gamma=0.95, train_freq=1, target_update_interval=200,
    exploration_fraction=0.3, exploration_final_eps=0.05,
    policy_kwargs=dict(net_arch=[128, 64]),
    verbose=1, device='cpu', seed=42,
)
router_v4_ddqn.learn(total_timesteps=ROUTER_TOTAL_TIMESTEPS, progress_bar=True)

router_v4_ddqn_path = MODEL_DIR / 'crypto_hrl_router_v4_ddqn_fix1004.zip'
router_v4_ddqn.save(str(router_v4_ddqn_path))
print(f'[DDQN] 라우터 저장: {router_v4_ddqn_path.name}')

In [ ]:
# 선택 분포 + 성과 비교: DQN 매칭 vs DDQN 매칭 vs β=30 단독 (α=0.01)
ALPHA = 0.01

def route_distribution(router_model, buckets):
    env = MacroRouterEnvV4(buckets=buckets, pool_models=[pool_models_v4[b] for b in POOL_BETAS],
                           route_buckets=ROUTE_BUCKETS, macro_window=MACRO_WINDOW, top_k=TOP_K,
                           episode_progress_mode='constant', **env_kwargs)
    obs, _ = env.reset()
    done, chosen = False, []
    while not done:
        a, _ = router_model.predict(obs, deterministic=True)
        chosen.append(POOL_BETAS[int(a)])
        obs, _, done, _, _ = env.step(a)
    return pd.Series(chosen).value_counts().to_dict()


def run_router_smoothed(router_model, buckets, alpha=ALPHA):
    env = MacroRouterEnvV4(buckets=buckets, pool_models=[pool_models_v4[b] for b in POOL_BETAS],
                           route_buckets=ROUTE_BUCKETS, macro_window=MACRO_WINDOW, top_k=TOP_K,
                           episode_progress_mode='constant', **env_kwargs)
    obs, _ = env.reset()
    inner, done = env.inner, False
    while not done:
        r, _ = router_model.predict(obs, deterministic=True)
        model = env.pool_models[int(r)]
        done_inner = False
        for _ in range(env.route_buckets):
            a, _ = model.predict(inner._get_obs(), deterministic=True)
            target_w, rho, top_idx = inner.action_to_weights(a)
            w = (1 - alpha) * inner.weights + alpha * target_w
            _, _, done_inner, _, info = inner.step_with_weights(w, extra_info={'rho': rho, 'selected': top_idx.tolist()})
            env._nav_history.append(info['portfolio_value'])
            if done_inner:
                break
        obs = env._macro_obs()
        done = done_inner or inner.t >= len(inner.buckets) - 1
    return np.array(inner.value_history)


def run_solo_smoothed(beta, buckets, alpha=ALPHA):
    env = SparseGatedCryptoPortfolioEnv(buckets=buckets, top_k=TOP_K, **env_kwargs)
    obs, _ = env.reset()
    done = False
    while not done:
        a, _ = pool_models_v4[beta].predict(obs, deterministic=True)
        target_w, rho, top_idx = env.action_to_weights(a)
        w = (1 - alpha) * env.weights + alpha * target_w
        obs, _, done, _, _ = env.step_with_weights(w, extra_info={'rho': rho, 'selected': top_idx.tolist()})
    return np.array(env.value_history)


rows = {}
for name, router in [('DQN 매칭', router_v4_matched), ('DDQN 매칭', router_v4_ddqn)]:
    for split, bk in [('valid', valid_buckets), ('test', test_buckets)]:
        dist = route_distribution(router, bk)
        p = perf_num(run_router_smoothed(router, bk))
        rows[(name, split)] = {**p, 'route_dist': str(dist)}
        print(f'{name} {split}: 선택 분포 {dist} | 수익 {p["ret"]:+.2%} 샤프 {p["sharpe"]:.3f} MDD {p["mdd"]:.2%}')
for split, bk in [('valid', valid_buckets), ('test', test_buckets)]:
    for beta in POOL_BETAS:
        p = perf_num(run_solo_smoothed(beta, bk))
        rows[(f'β={beta} 단독', split)] = {**p, 'route_dist': ''}
print()
res = pd.DataFrame(rows).T
print(res.to_string())
res.to_csv(CRYPTO_DIR / 'crypto_hrl_v4_dqn_ddqn_router_fix1004.csv', encoding='utf-8-sig')
print('\n저장 완료: crypto_hrl_v4_dqn_ddqn_router_fix1004.csv')
